In [ ]:
%load_ext autoreload
%autoreload 2

import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS

from trunx.gp3.model_inputs import State
from trunx.gp3.PG3_model_impl import prepare_data
from trunx.gp3.run_3pg import run_3pg

In [ ]:
def test_model(
    climate,
    site,
    species,
    n_species: int,
    obs_DBH: jnp.ndarray | None = None,
    obs_times: jnp.ndarray | None = None,
    initial_state: State | None = None,
    fixed_params=None,
):
    """A test model for parameter estimation using HMC."""
    assert fixed_params is not None
    alphaCx = numpyro.sample("alphaCx", dist.LogNormal(jnp.log(0.05), 0.5))
    CoeffCond = numpyro.sample("CoeffCond", dist.LogNormal(jnp.log(0.05), 0.5))

    Y = numpyro.sample("Y", dist.Normal(0.47, 0.05))
    gammaF0 = numpyro.sample("gammaF0", dist.LogNormal(jnp.log(0.001), 0.3))
    gammaF1 = numpyro.sample("gammaF1", dist.LogNormal(jnp.log(0.02), 0.3))
    tgammaF = numpyro.sample("tgammaF", dist.LogNormal(jnp.log(60.0), 0.5))
    tRho = numpyro.sample("tRho", dist.Normal(jnp.log(1.0), 0.02))

    params = fixed_params._replace(
        alphaCx=alphaCx,
        CoeffCond=CoeffCond,
        Y=Y,
        gammaF0=gammaF0,
        gammaF1=gammaF1,
        tgammaF=tgammaF,
        tRho=tRho,
    )

    _, outputs = run_3pg(initial_state, climate, params, site, species, n_species)

    pred_DBH = outputs["DBH"][obs_times] if obs_times is not None else outputs["DBH"]
    sigma_DBH = numpyro.sample("sigma_DBH", dist.HalfNormal(1.0))
    numpyro.sample("obs_DBH", dist.StudentT(df=3, loc=pred_DBH, scale=sigma_DBH), obs=obs_DBH)


file_path = "./data/data_sspecies_nothinning.xlsx"
initial_state, climate, fixed_params, site_data, species_data, n_species = prepare_data(file_path)

obs_times = jnp.array([12, 24, 36, 48, 60, 72, 84, 96, 108, 120, 132])
obs_DBH = jnp.array([14, 14.8, 15.2, 15.9, 15.8, 16.1, 17.3, 17.8, 18.5, 18.8, 19.2])

kernel = NUTS(test_model)
mcmc = MCMC(kernel, num_warmup=200, num_samples=200, num_chains=2)

mcmc.run(
    jax.random.PRNGKey(42),
    climate=climate,
    site=site_data,
    species=species_data,
    n_species=n_species,
    obs_DBH=obs_DBH,
    obs_times=obs_times,
    initial_state=initial_state,
    fixed_params=fixed_params,
)

In [ ]:
mcmc.print_summary()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

samples = mcmc.get_samples()
samples_df = pd.DataFrame(samples)

cols = 4
rows = (len(samples_df.columns) + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(15, 4 * rows))

axes = axes.flatten()

for i, col in enumerate(samples_df.columns):
    sns.histplot(
        data=samples_df, x=col, ax=axes[i], kde=True, edgecolor="black", alpha=0.7, bins=30
    )
    axes[i].set_title(col, fontsize=12, fontweight="bold")
    axes[i].set_xlabel("Value", fontsize=10)
    axes[i].set_ylabel("Frequency", fontsize=10)
    axes[i].grid(True, alpha=0.3)

for i in range(len(samples_df.columns), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()